In [5]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, Verbose
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [6]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [7]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [8]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model(
    save_name="tmp-tests", tokenizer=TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer"), 
    device="cuda", depth=6, head_dim=128*2, context_size=4096*1, nb_heads_mult=5.0)
model.show_infos()

loading the tokenizer from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=2, n_kv_head=2, n_embd=512, window_pattern='SSSL')
20_512_972 total params (embeding: 1_310_720 | last layer: 327_680 | transformer: 18_874_560)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [5]:
ID_1, _ = model.save("un_trained")

saving the tokenizer to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0001_un_trained/history.json


In [9]:
try:
    if "dataset" in globals() : raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)
except FileExistsError: pass

In [10]:
model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=1, timeLimite=999_999, verbose=Verbose.liveProgress)


starting epoch: 1


W0310 17:48:44.117000 835795 torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode


finished batches in 12 min 59.3 sec                  
finished epoches in 12 min 59.3 sec
trained on: 863 batch (13793 chuncks) in 12 min 59.3 sec
 -> 1.11 batch/sec | 17.70 chuncks/sec
 -> CE: 1 | PPL: 2.72 | top-1: 75.53%


In [8]:
ID_2, _ = model.save("trained-1epoches")

saving the tokenizer to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0002_trained-1epoches/history.json


In [9]:
new_model = Model.load("tmp-tests", ID_2, torch.device("cuda:0"))

loading the tokenizer from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
loading the historique from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0002_trained-1epoches/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490


In [10]:
new_model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=1, timeLimite=999_999, verbose=Verbose.liveProgress)


starting epoch: 1
finished batches in 2 min 54.1 sec              
finished epoches in 2 min 54.1 sec
trained on: 863 batch (13793 chuncks) in 2 min 54.1 sec
 -> 4.96 batch/sec | 79.22 chuncks/sec
 -> CE: 0.8255 | PPL: 2.283 | top-1: 79.44%


In [ ]:
ID_3, _ = new_model.save("trained-2epoches")

In [11]:
for i in range(3, 5+1):
    new_model.train(
        dataset=dataset, batch_size=16, 
        nbEpoches=1, timeLimite=999_999, verbose=Verbose.liveProgress)
    _, _ = new_model.save(f"trained-{i}epoches")


starting epoch: 1
finished batches in 2 min 55.5 sec              
finished epoches in 2 min 55.5 sec
trained on: 863 batch (13793 chuncks) in 2 min 55.5 sec
 -> 4.92 batch/sec | 78.59 chuncks/sec
 -> CE: 0.7872 | PPL: 2.197 | top-1: 80.61%
saving the tokenizer to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0003_trained-3epoches/history.json

starting epoch: 1
finished batches in 3 min 1.9 sec               
finished epoches in 3 min 1.9 sec
trained on: 863 batch (13793 chuncks) in 3 min 1.9 sec
 -> 4.74 batch/sec | 75.83 chuncks/sec
 -> CE: 0.7638 | PPL: 2.147 | top-1: 81.36%
saving the tokenizer to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE